# Limpieza y Análisis de Datos de Criminalidad por Cantón en Costa Rica

Este notebook realiza el procesamiento completo de datos de criminalidad:
- Carga datos del OIJ (archivos XLS/XLSX) y GeoJSON de cantones
- Normaliza nombres de cantones y provincias
- Limpia datos inconsistentes
- Agrega delitos por cantón y calcula top 3 delitos
- Une datos con geometría geoespacial
- Genera mapa choropleth interactivo con Folium
- Exporta resultados limpios en CSV y GeoPackage

In [1]:
import pathlib

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from unidecode import unidecode

pd.options.display.max_columns = None
pd.options.display.max_rows = 100


def normalize_canton_name(name: str) -> str:
    """Normaliza nombres de cantones: elimina tildes, convierte a minúsculas y elimina espacios extra."""
    if pd.isna(name):
        return ""
    s = str(name).strip().lower()
    s = unidecode(s)
    s = " ".join(s.split())
    return s

In [2]:
def load_oij_data(oij_dir):
    """Carga todos los archivos XLSX/XLS en la carpeta oij_dir y los concatena."""
    oij_files = list(pathlib.Path(oij_dir).glob("*.xls*"))
    df_oij_list = []
    for file in oij_files:
        df = pd.read_excel(file, engine='openpyxl' if file.suffix == '.xlsx' else None)
        df_oij_list.append(df)
    df_oij = pd.concat(df_oij_list, ignore_index=True) if df_oij_list else pd.DataFrame()
    return df_oij

In [3]:
def load_geojson_data(geo_path):
    """Carga el archivo GeoJSON usando geopandas o pandas si falla."""
    try:
        gdf_cantones = gpd.read_file(geo_path)
    except Exception as e:
        print(f"Error con geopandas: {e}. Intentando con pandas.")
        df = pd.read_json(geo_path)
        # Asumir que es GeoJSON con features
        if 'features' in df.columns:
            gdf_cantones = gpd.GeoDataFrame.from_features(df['features'])
        else:
            raise ValueError("No se pudo cargar el GeoJSON.")
    return gdf_cantones

In [4]:
def explore_dataset(df, name):
    """Imprime información básica del dataset para exploración inicial."""
    print(f"\n=== Exploración de {name} ===")
    print("Shape:", df.shape)
    print("Tipos de datos:\n", df.dtypes)
    print("Primeras filas:\n", df.head(5))
    print("Valores nulos por columna:\n", df.isna().sum())

In [5]:
# Funciones separadas en celdas anteriores

In [6]:
# Definir rutas relativas al directorio actual del notebook.
base_dir = pathlib.Path.cwd() / "Proyecto"
oij_dir = base_dir / "Datos_poder_judicial"
geo_path = base_dir / "Datos_GeoJSON" / "Cantones_GeoJSON.json"

print("Base dir:", base_dir)
print("OIJ dir:", oij_dir)
print("Geo path:", geo_path)

print("Cargando datasets...")
df_oij = load_oij_data(oij_dir)
gdf_cantones = load_geojson_data(geo_path)

print("Datos de OIJ cargados correctamente.")
print("Datos de GeoJSON cargados correctamente.")

explore_dataset(df_oij, "OIJ")
explore_dataset(gdf_cantones, "GeoDataFrame de cantones")

# Ver columnas disponibles para cantones
print("\nColumnas en df_oij:", df_oij.columns.tolist())
print("Columnas en gdf_cantones:", gdf_cantones.columns.tolist())

# Variables clave para coincidencia con el análisis
provincia_col = "Provincia"
canton_col = "Canton"
geo_prov_col = "NAME_1"
geo_cant_col = "NAME_2"

Base dir: /workspaces/Analisis-de-datos-II/Proyecto
OIJ dir: /workspaces/Analisis-de-datos-II/Proyecto/Datos_poder_judicial
Geo path: /workspaces/Analisis-de-datos-II/Proyecto/Datos_GeoJSON/Cantones_GeoJSON.json
Cargando datasets...
Datos de OIJ cargados correctamente.
Datos de GeoJSON cargados correctamente.

=== Exploración de OIJ ===
Shape: (599604, 13)
Tipos de datos:
 Delito             str
SubDelito          str
Fecha           object
Victima            str
SubVictima         str
Edad               str
Genero             str
Nacionalidad       str
Provincia          str
Canton             str
Distrito           str
Hora               str
Sexo               str
dtype: object
Primeras filas:
              Delito SubDelito       Fecha   Victima              SubVictima  \
0  ROBO DE VEHICULO  DESCUIDO  2017-12-15  VEHICULO     MICROBUS [VEHICULO]   
1  ROBO DE VEHICULO  DESCUIDO  2017-12-15  VEHICULO  MOTOCICLETA [VEHICULO]   
2  ROBO DE VEHICULO  DESCUIDO  2017-12-15  VEHICULO  MOTO

In [7]:
# Limpiar espacios dentro de los nombres de provincia y cantón en df_oij
for col in ["Provincia", "Canton"]:
    if col in df_oij.columns:
        df_oij[col] = df_oij[col].astype(str).str.replace(r"\s+", "", regex=True)

# Reemplazar nombres específicos para coincidir con el formato esperado
if "Canton" in df_oij.columns:
    df_oij["Canton"] = df_oij["Canton"].replace({
        "VASQUEZDECORONADO": "VAZQUEZDECORONADO",
        "LEONCORTES": "LEONCORTESCASTRO",
        "MONTEVERDE": "PUNTARENAS",  # Asumir que Monteverde pertenece a Puntarenas
        "PUERTOJIMENEZ": "GOLFITO",  # Puerto Jiménez pertenece a Golfito
        "VALVERDEVEGA": "SARCHI",  # Valverde Vega pertenece a Sarchí
    })

print("Limpieza realizada en df_oij:")
print(df_oij[["Provincia", "Canton"]].head(10))

Limpieza realizada en df_oij:
    Provincia        Canton
0  GUANACASTE       BAGACES
1     SANJOSE       MORAVIA
2     SANJOSE       MORAVIA
3     CARTAGO       LAUNION
4     SANJOSE  PEREZZELEDON
5     HEREDIA     SARAPIQUI
6  GUANACASTE       LIBERIA
7     CARTAGO      ELGUARCO
8    ALAJUELA      ALAJUELA
9    ALAJUELA     LOSCHILES


In [8]:
# Confirmar coincidencias entre gdf_cantones y df_oij usando las columnas reales.
provincia_col = "Provincia"
canton_col = "Canton"
geo_prov_col = "NAME_1"
geo_cant_col = "NAME_2"

print("Columnas df_oij:", df_oij.columns.tolist())
print("Columnas gdf_cantones:", gdf_cantones.columns.tolist())

# Normalizar columnas clave con normalize_canton_name
for col in [provincia_col, canton_col]:
    if col in df_oij.columns:
        df_oij[f"{col}_normalizado"] = df_oij[col].apply(normalize_canton_name)
    else:
        raise KeyError(f"Columna esperada no encontrada en df_oij: {col}")

for col in [geo_prov_col, geo_cant_col]:
    if col in gdf_cantones.columns:
        gdf_cantones[f"{col}_normalizado"] = gdf_cantones[col].apply(normalize_canton_name)
    else:
        raise KeyError(f"Columna esperada no encontrada en gdf_cantones: {col}")

prov_df = set(df_oij[f"{provincia_col}_normalizado"].dropna().unique())
prov_geo = set(gdf_cantones[f"{geo_prov_col}_normalizado"].dropna().unique())

cant_df = set(df_oij[f"{canton_col}_normalizado"].dropna().unique())
cant_geo = set(gdf_cantones[f"{geo_cant_col}_normalizado"].dropna().unique())

print(f"\nProvincia: df_oij={len(prov_df)}, gdf_cantones={len(prov_geo)}, comunes={len(prov_df & prov_geo)}")
print(f"Cantones: df_oij={len(cant_df)}, gdf_cantones={len(cant_geo)}, comunes={len(cant_df & cant_geo)}")

print("\nProvincias solo en df_oij:")
print(sorted(prov_df - prov_geo))
print("\nProvincias solo en gdf_cantones:")
print(sorted(prov_geo - prov_df))

print("\nCantones solo en df_oij:")
print(sorted(cant_df - cant_geo)[:100])
print("\nCantones solo en gdf_cantones:")
print(sorted(cant_geo - cant_df)[:100])

print("\nPrimeros 20 provincias comunes:")
print(sorted(list(prov_df & prov_geo))[:20])
print("\nPrimeros 20 cantones comunes:")
print(sorted(list(cant_df & cant_geo))[:20])

Columnas df_oij: ['Delito', 'SubDelito', 'Fecha', 'Victima', 'SubVictima', 'Edad', 'Genero', 'Nacionalidad', 'Provincia', 'Canton', 'Distrito', 'Hora', 'Sexo']
Columnas gdf_cantones: ['GID_2', 'GID_0', 'COUNTRY', 'GID_1', 'NAME_1', 'NL_NAME_1', 'NAME_2', 'VARNAME_2', 'NL_NAME_2', 'TYPE_2', 'ENGTYPE_2', 'CC_2', 'HASC_2', 'geometry']

Provincia: df_oij=8, gdf_cantones=7, comunes=7
Cantones: df_oij=84, gdf_cantones=82, comunes=82

Provincias solo en df_oij:
['desconocido']

Provincias solo en gdf_cantones:
[]

Cantones solo en df_oij:
['desconocido', 'puertojimenez']

Cantones solo en gdf_cantones:
[]

Primeros 20 provincias comunes:
['alajuela', 'cartago', 'guanacaste', 'heredia', 'limon', 'puntarenas', 'sanjose']

Primeros 20 cantones comunes:
['abangares', 'acosta', 'alajuela', 'alajuelita', 'alvarado', 'aserri', 'atenas', 'bagaces', 'barva', 'belen', 'buenosaires', 'canas', 'carrillo', 'cartago', 'corredores', 'cotobrus', 'curridabat', 'desamparados', 'dota', 'elguarco']


In [9]:
# ============================================================================
# PREPARACIÓN: CREAR COLUMNAS NORMALIZADAS
# ============================================================================
# Antes de limpiar, créamos columnas normalizadas de Provincia y Canton

print("=== CREACIÓN DE COLUMNAS NORMALIZADAS ===\n")

# Agregar columnas normalizadas usando normalize_canton_name
df_oij['Provincia_normalizado'] = df_oij['Provincia'].apply(normalize_canton_name).str.upper()
df_oij['Canton_normalizado'] = df_oij['Canton'].apply(normalize_canton_name).str.upper()

# Recrear en gdf_cantones
gdf_cantones['NAME_1_normalizado'] = gdf_cantones['NAME_1'].apply(normalize_canton_name).str.upper()
gdf_cantones['NAME_2_normalizado'] = gdf_cantones['NAME_2'].apply(normalize_canton_name).str.upper()

print(f"Columnas normalizadas creadas en df_oij y gdf_cantones")
print(f"Provincias únicas en df_oij: {df_oij['Provincia_normalizado'].nunique()}")
print(f"Cantones únicos en df_oij: {df_oij['Canton_normalizado'].nunique()}")
print(f"\nPrimeros ejemplos de normalización en df_oij:")
print(df_oij[['Provincia', 'Provincia_normalizado', 'Canton', 'Canton_normalizado']].head())

=== CREACIÓN DE COLUMNAS NORMALIZADAS ===

Columnas normalizadas creadas en df_oij y gdf_cantones
Provincias únicas en df_oij: 8
Cantones únicos en df_oij: 84

Primeros ejemplos de normalización en df_oij:
    Provincia Provincia_normalizado        Canton Canton_normalizado
0  GUANACASTE            GUANACASTE       BAGACES            BAGACES
1     SANJOSE               SANJOSE       MORAVIA            MORAVIA
2     SANJOSE               SANJOSE       MORAVIA            MORAVIA
3     CARTAGO               CARTAGO       LAUNION            LAUNION
4     SANJOSE               SANJOSE  PEREZZELEDON       PEREZZELEDON


In [10]:
# ============================================================================
# LIMPIEZA DE CANTONES PROBLEMÁTICOS
# ============================================================================
# Este bloque resuelve inconsistencias en la normalización de cantones

print("=== LIMPIEZA DE CANTONES PROBLEMÁTICOS ===\n")

# PASO 0: Remover espacios en las columnas normalizadas
df_oij['Provincia_normalizado'] = df_oij['Provincia_normalizado'].str.replace(' ', '', regex=False)
df_oij['Canton_normalizado'] = df_oij['Canton_normalizado'].str.replace(' ', '', regex=False)
gdf_cantones['NAME_1_normalizado'] = gdf_cantones['NAME_1_normalizado'].str.replace(' ', '', regex=False)
gdf_cantones['NAME_2_normalizado'] = gdf_cantones['NAME_2_normalizado'].str.replace(' ', '', regex=False)

print("✓ Espacios removidos de columnas normalizadas")

# Filas iniciales antes de limpieza
filas_antes = len(df_oij)
print(f"Filas totales en df_oij antes: {filas_antes}")

# 1. Eliminar filas con 'desconocido' en provincia o cantón
df_oij = df_oij[
    (df_oij['Provincia_normalizado'] != 'DESCONOCIDO') & 
    (df_oij['Canton_normalizado'] != 'DESCONOCIDO')
]
filas_desconocido = filas_antes - len(df_oij)
print(f"Filas eliminadas por 'desconocido': {filas_desconocido}")

# 2. Verificación post-limpieza
print(f"\nCantones únicos tras limpieza: {df_oij['Canton_normalizado'].nunique()}")
print(f"Filas totales tras limpieza: {len(df_oij)}")

# 3. Mostrar cantones solo en df_oij (antes del merge)
cant_df_clean = set(df_oij['Canton_normalizado'].dropna().unique())
cant_geo_clean = set(gdf_cantones['NAME_2_normalizado'].dropna().unique())

cantones_solo_oij = sorted(cant_df_clean - cant_geo_clean)
cantones_solo_geo = sorted(cant_geo_clean - cant_df_clean)

print(f"\nCantones solo en df_oij (posibles desajustes): {len(cantones_solo_oij)}")
if cantones_solo_oij:
    print(cantones_solo_oij)
else:
    print("✓ LISTA VACÍA - Todos los cantones coinciden después de correcciones")

print(f"\nCantones solo en gdf_cantones (sin delitos registrados): {len(cantones_solo_geo)}")



=== LIMPIEZA DE CANTONES PROBLEMÁTICOS ===

✓ Espacios removidos de columnas normalizadas
Filas totales en df_oij antes: 599604
Filas eliminadas por 'desconocido': 5271

Cantones únicos tras limpieza: 83
Filas totales tras limpieza: 594333

Cantones solo en df_oij (posibles desajustes): 1
['PUERTOJIMENEZ']

Cantones solo en gdf_cantones (sin delitos registrados): 0


In [11]:
# ============================================================================
# PASO 1: AGREGAR DELITOS POR CANTÓN Y TOP 3 DELITOS
# ============================================================================
# Agregar conteo total de filas por cantón en df_oij

delitos_por_canton = (
    df_oij.groupby("Canton_normalizado", as_index=False)
    .size()
    .rename(columns={"size": "total_delitos"})
)

# Calcular los top 3 delitos por cantón como listas de strings
_top_delitos_por_canton = (
    df_oij.groupby(["Canton_normalizado", "Delito"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .sort_values(["Canton_normalizado", "count"], ascending=[True, False])
)
_top_delitos_por_canton["top_delito_str"] = _top_delitos_por_canton.apply(
    lambda row: f"{row['Delito']}: {int(row['count']):,}", axis=1
)

top_delitos_map = (
    _top_delitos_por_canton.groupby("Canton_normalizado")["top_delito_str"]
    .apply(lambda values: list(values.head(3)))
    .to_dict()
)

# Merge left del GeoDataFrame con el conteo de delitos

gdf_delitos = gdf_cantones.merge(
    delitos_por_canton,
    left_on="NAME_2_normalizado",
    right_on="Canton_normalizado",
    how="left",
)

gdf_delitos["total_delitos"] = gdf_delitos["total_delitos"].fillna(0).astype(int)

# Generar columna top_delitos como lista de strings lista para mostrar

gdf_delitos["top_delitos"] = gdf_delitos["NAME_2_normalizado"].map(top_delitos_map)

gdf_delitos["top_delitos"] = gdf_delitos["top_delitos"].apply(lambda x: x if isinstance(x, list) else [])

# Verificación del resultado

total_cantones = len(gdf_delitos)
cantones_con_delitos = int((gdf_delitos["total_delitos"] > 0).sum())
cantones_sin_delitos = int((gdf_delitos["total_delitos"] == 0).sum())
suma_total_delitos = int(gdf_delitos["total_delitos"].sum())

print(f"Total de cantones en el GeoDataFrame: {total_cantones}")
print(f"Cantones con delitos: {cantones_con_delitos}")
print(f"Cantones sin delitos: {cantones_sin_delitos}")
print(f"Suma total de delitos en el mapa: {suma_total_delitos:,}")

Total de cantones en el GeoDataFrame: 82
Cantones con delitos: 82
Cantones sin delitos: 0
Suma total de delitos en el mapa: 594,035


In [13]:
# ============================================================================
# GENERACIÓN DEL MAPA CHOROPLETH CON FOLIUM
# ============================================================================

import folium
from folium import plugins
import json
from shapely.geometry import mapping

print("=== GENERACIÓN DEL MAPA CHOROPLETH ===\n")

# Parámetros del mapa
centro_costa_rica = [9.7489, -83.7534]
zoom_inicial = 8

# Crear mapa base centrado en Costa Rica con tiles de CartoDB Positron
mapa = folium.Map(
    location=centro_costa_rica,
    zoom_start=zoom_inicial,
    tiles=None  # Sin tiles por defecto
)

# Agregar tile layer de CartoDB Positron
folium.TileLayer(
    tiles='https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> &copy; <a href="https://carto.com/">CARTO</a>',
    name='CartoDB Positron'
).add_to(mapa)

print(f"Mapa creado centrado en: {centro_costa_rica} con tiles CartoDB Positron")

# Preparar GeoJSON para Folium
# Convertir a formato que Folium pueda leer
geo_json_str = gdf_delitos.to_json()
geo_json_data = json.loads(geo_json_str)

# Crear capa de Choropleth
choropleth = folium.Choropleth(
    geo_data=geo_json_data,
    name='Delitos por Cantón',
    data=gdf_delitos,
    columns=['NAME_2_normalizado', 'total_delitos'],
    key_on='feature.properties.NAME_2_normalizado',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name='Total de delitos 2017–presente',
    nan_fill_color='lightgray'
)
choropleth.add_to(mapa)

print("Capa Choropleth agregada al mapa")

# Agregar tooltips y popups usando GeoJson con funciones personalizadas
def crear_propiedades_enriquecidas(row):
    """Crea un diccionario con todas las propiedades para cada feature"""
    return {
        'NAME_1': row.get('NAME_1', ''),
        'NAME_2': row.get('NAME_2', ''),
        'NAME_1_normalizado': row.get('NAME_1_normalizado', ''),
        'NAME_2_normalizado': row.get('NAME_2_normalizado', ''),
        'total_delitos': int(row.get('total_delitos', 0)),
        'top_delitos': str(row.get('top_delitos', []))
    }

# Re-crear GeoJSON con propiedades enriquecidas
geo_features = []
for idx, row in gdf_delitos.iterrows():
    feature = {
        'type': 'Feature',
        'geometry': mapping(row.geometry),
        'properties': crear_propiedades_enriquecidas(row)
    }
    geo_features.append(feature)

geo_json_enriquecido = {
    'type': 'FeatureCollection',
    'features': geo_features
}

# Agregar GeoJson con tooltips y popups
folium.GeoJson(
    geo_json_enriquecido,
    style_function=lambda x: {
        'fillOpacity': 0,
        'color': 'transparent'
    },
    highlight_function=lambda x: {
        'fillOpacity': 0.3,
        'color': 'black'
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAME_2', 'NAME_1', 'total_delitos'],
        aliases=['Cantón:', 'Provincia:', 'Total delitos:'],
        style="background-color: #f1f1f1; border-radius: 5px; box-shadow: 2px 2px 6px rgba(0,0,0,0.3);"
    ),
    popup=folium.GeoJsonPopup(
        fields=['NAME_2', 'NAME_1', 'total_delitos'],
        aliases=['Cantón:', 'Provincia:', 'Total de delitos:'],
        max_width=300
    )
).add_to(mapa)

print("Tooltips y popups agregados")

# Agregar control de capas
folium.LayerControl().add_to(mapa)

# Guardar el mapa
ruta_mapa = base_dir / "mapa_criminalidad_cantones.html"
mapa.save(str(ruta_mapa))

print(f"\n✓ Mapa guardado en: {ruta_mapa}")
print(f"\nResumen del mapa:")
print(f"- Cantones totales: {len(gdf_delitos)}")
con_match = len(gdf_delitos)
sin_match = total_cantones - con_match
print(f"- Cantones con delitos: {con_match}")
print(f"- Cantones sin delitos: {sin_match}")
print(f"- Total de delitos visualizados: {gdf_delitos['total_delitos'].sum():,}")

=== GENERACIÓN DEL MAPA CHOROPLETH ===

Mapa creado centrado en: [9.7489, -83.7534] con tiles CartoDB Positron
Capa Choropleth agregada al mapa
Tooltips y popups agregados

✓ Mapa guardado en: /workspaces/Analisis-de-datos-II/Proyecto/mapa_criminalidad_cantones.html

Resumen del mapa:
- Cantones totales: 82
- Cantones con delitos: 82
- Cantones sin delitos: 0
- Total de delitos visualizados: 594,035
